In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import requests

In [ ]:
def get_full_60_pick_mock():
    url = "https://www.nbadraft.net/nba-mock-drafts/"
    
    # Mimic a real browser to avoid 403 Forbidden errors
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        r = requests.get(url, headers=headers)
        r.raise_for_status()
        
        # Read ALL tables from the page
        # valid_dfs will store the clean versions of the tables we find
        all_dfs = pd.read_html(r.text)
        valid_dfs = []
        
        for df in all_dfs:
            # Filter for actual draft tables
            # We check if the table has the 'Player' and 'Team' columns.
            # Some helper tables on the site (like "Last Updated") don't have these.
            if 'Player' in df.columns and 'Team' in df.columns:
                
                # Clean up: Drop rows that are just repeated headers (common in these tables)
                clean_df = df[df['Player'] != 'Player'].copy()
                
                # Select only the columns we care about
                # (NBADraft.net sometimes has extra empty columns)
                target_cols = ['Team', 'Player', 'Height', 'Weight', 'Position', 'School']
                
                # Only keep columns that actually exist in this specific table
                actual_cols = [c for c in target_cols if c in clean_df.columns]
                clean_df = clean_df[actual_cols]
                
                valid_dfs.append(clean_df)

        # Combine them
        if not valid_dfs:
            print("No valid draft tables found.")
            return None
            
        # Stack Round 1 and Round 2 on top of each other
        full_draft = pd.concat(valid_dfs, ignore_index=True)
        
        # Add the Pick Number (1-60)
        # We assume the order scraped is the order of the draft
        full_draft.insert(0, 'Pick', range(1, len(full_draft) + 1))
        
        return full_draft

    except Exception as e:
        print(f"Error: {e}")
        return None

# --- Run it ---
df_2026 = get_full_60_pick_mock()

In [ ]:
# df_2026.to_csv("pred_picks.csv")